# Vanguard Website A/B Test Analysis

## Project Overview

Vanguard conducted an A/B test comparing its traditional online process (**Control**) with a redesigned interface (**Test**).

Both versions follow the same process:

`start → step_1 → step_2 → step_3 → confirm`

The objective of this analysis is to evaluate whether the redesigned experience performs better using the following KPIs:

- **Completion rate**
- **Time spent on each process step**
- **Error rate**

The analysis also tests whether:

1. The Test version has a significantly higher completion rate than the Control version.
2. The relative improvement in completion rate exceeds Vanguard's required **5% cost-effectiveness threshold**.

# 1. Setup and Data Loading

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest

DATA_DIR = Path("../data/raw")

## 1.1 Load the datasets

In [ ]:
# Client profiles
client_path = DATA_DIR / "df_final_demo.txt"
clients_df = pd.read_csv(client_path)

# Digital footprint data
web_part1_path = DATA_DIR / "df_final_web_data_pt_1.txt"
web_part2_path = DATA_DIR / "df_final_web_data_pt_2.txt"

web_part1_df = pd.read_csv(web_part1_path)
web_part2_df = pd.read_csv(web_part2_path)

web_df = pd.concat(
    [web_part1_df, web_part2_df],
    axis = 0,
    ignore_index = True
)

# Experiment roster
experiment_path = DATA_DIR / "df_final_experiment_clients.txt"
experiment_df = pd.read_csv(experiment_path)

In [3]:
clients_df.head()

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0


In [4]:
web_df.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [5]:
experiment_df.head()

,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control


# 2. Data Cleaning

## 2.1 Client profiles

In [6]:
# Check missing values
clients_df.isnull().sum()

client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64

In [7]:
# Remove rows with substantial missing data
clients_df = clients_df.dropna(thresh = 7)

# Fill the remaining missing age value using the mean
clients_df['clnt_age'].fillna(
    clients_df['clnt_age'].mean(),
    inplace = True
)

clients_df.isnull().sum()

C:\Users\ramya\AppData\Local\Temp\ipykernel_33056\2681486373.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  clients_df['clnt_age'].fillna(


client_id           0
clnt_tenure_yr      0
clnt_tenure_mnth    0
clnt_age            0
gendr               0
num_accts           0
bal                 0
calls_6_mnth        0
logons_6_mnth       0
dtype: int64

In [8]:
# Review unique values
clients_df.nunique()

client_id           70595
clnt_tenure_yr         54
clnt_tenure_mnth      482
clnt_age              166
gendr                   4
num_accts               8
bal                 70328
calls_6_mnth            8
logons_6_mnth           9
dtype: int64

In [9]:
# Standardise gender values
clients_df['gendr'] = clients_df['gendr'].map({
    "X":"U",
    "F":"F",
    "M":"M",
    "U":"U"
})

clients_df['gendr'].value_counts()

gendr
U    24125
M    23724
F    22746
Name: count, dtype: int64

In [10]:
# Convert selected columns to integer
columns = [
    "clnt_tenure_yr",
    "clnt_tenure_mnth",
    "clnt_age",
    "num_accts",
    "calls_6_mnth",
    "logons_6_mnth"
]

for column in columns:
    clients_df[column] = clients_df[column].astype(int)

clients_df.dtypes

client_id             int64
clnt_tenure_yr        int64
clnt_tenure_mnth      int64
clnt_age              int64
gendr                object
num_accts             int64
bal                 float64
calls_6_mnth          int64
logons_6_mnth         int64
dtype: object

In [11]:
# Rename columns for readability
clients_df.columns = [
    'client_id',
    'client_tenure_yr',
    'client_tenure_month',
    'client_age',
    'gender',
    'num_accounts',
    'balance',
    'calls_6_month',
    'logons_6_month'
]

clients_df.head()

,client_id,client_tenure_yr,client_tenure_month,client_age,gender,num_accounts,balance,calls_6_month,logons_6_month
0,836976,6,73,60,U,2,45105.30,6,9
1,2304905,7,94,58,U,2,110860.30,6,9
2,1439522,5,64,32,U,2,52467.79,6,9
3,1562045,16,198,49,M,2,67454.65,3,6
4,5126305,12,145,33,F,2,103671.75,0,3


## 2.2 Digital footprint data

In [12]:
web_df.isnull().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

In [13]:
web_df['process_step'].value_counts()

process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

In [14]:
web_df.dtypes

client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
dtype: object

## 2.3 Experiment roster

In [15]:
experiment_df.isnull().sum()

client_id        0
Variation    20109
dtype: int64

In [16]:
experiment_df['Variation'].value_counts()

Variation
Test       26968
Control    23532
Name: count, dtype: int64

In [17]:
experiment_df.dtypes

client_id     int64
Variation    object
dtype: object

# 3. Client Behavior Analysis

This section first reviews the overall client population, then focuses on clients assigned to the A/B experiment.

## 3.1 Overall client profile

In [18]:
clients_df.describe()

,client_id,client_tenure_yr,client_tenure_month,client_age,num_accounts,balance,calls_6_month,logons_6_month
count,7.059500e+04,70595.000000,70595.000000,70595.000000,70595.000000,7.059500e+04,70595.000000,70595.000000
mean,5.005021e+06,12.052950,150.659367,46.180424,2.255528,1.474452e+05,3.382478,5.566740
std,2.877269e+06,6.871819,82.089854,15.600279,0.534997,3.015087e+05,2.236580,2.353286
min,1.690000e+02,2.000000,33.000000,13.000000,1.000000,1.378942e+04,0.000000,1.000000
25%,2.519604e+06,6.000000,82.000000,32.000000,2.000000,3.734683e+04,1.000000,4.000000
50%,5.016969e+06,11.000000,136.000000,47.000000,2.000000,6.333290e+04,3.000000,5.000000
75%,7.483064e+06,16.000000,192.000000,59.000000,2.000000,1.375449e+05,6.000000,7.000000
max,9.999839e+06,62.000000,749.000000,96.000000,8.000000,1.632004e+07,7.000000,9.000000


The descriptive statistics above provide an overview of client age, tenure, account holdings, balances, calls, and logons.

## 3.2 Experiment population

In [19]:
# Merge client profiles with experiment assignments
clients_experiment_df = pd.merge(
    clients_df,
    experiment_df,
    on = "client_id",
    how = "left"
)

clients_experiment_df.isna().sum()

client_id                  0
client_tenure_yr           0
client_tenure_month        0
client_age                 0
gender                     0
num_accounts               0
balance                    0
calls_6_month              0
logons_6_month             0
Variation              20107
dtype: int64

In [20]:
# Keep only clients assigned to Test or Control
clients_experiment_df.dropna(
    subset = ['Variation'],
    inplace = True
)

clients_experiment_df.describe()

,client_id,client_tenure_yr,client_tenure_month,client_age,num_accounts,balance,calls_6_month,logons_6_month
count,5.048800e+04,50488.000000,50488.000000,50488.000000,50488.000000,5.048800e+04,50488.000000,50488.000000
mean,5.006173e+06,12.031730,150.415485,47.058430,2.254575,1.495147e+05,3.093289,6.131873
std,2.877417e+06,6.860282,81.944830,15.527939,0.533671,3.020364e+05,2.187991,2.175423
min,5.550000e+02,2.000000,33.000000,17.000000,1.000000,2.378944e+04,0.000000,3.000000
25%,2.515700e+06,6.000000,82.000000,33.000000,2.000000,3.987841e+04,1.000000,4.000000
50%,5.025026e+06,11.000000,136.000000,48.000000,2.000000,6.573360e+04,3.000000,6.000000
75%,7.477918e+06,16.000000,192.000000,59.000000,2.000000,1.399565e+05,5.000000,8.000000
max,9.999832e+06,55.000000,669.000000,96.000000,7.000000,1.632004e+07,6.000000,9.000000


In [21]:
# Keep only web events belonging to experiment clients
web_experiment_df = web_df[
    web_df['client_id'].isin(
        clients_experiment_df['client_id']
    )
].copy()

web_experiment_df['date_time'] = pd.to_datetime(
    web_experiment_df['date_time']
)

print(
    "Unique experiment clients in web data:",
    web_experiment_df['client_id'].nunique()
)

print()
print(
    web_experiment_df['process_step'].value_counts()
)

Unique experiment clients in web data: 50488

process_step
start      104046
step_1      68412
step_2      56857
step_3      48677
confirm     43215
Name: count, dtype: int64


## 3.3 Age groups and experiment-group characteristics

In [22]:
# Create age groups
bins = [0,18,34,54,74,clients_df['client_age'].max()]
labels = ['0-18','19-34','35-54','55-74','>74']

clients_df['age_group'] = pd.cut(
    clients_df['client_age'],
    labels = labels,
    bins = bins,
    include_lowest = True
)

clients_experiment_df['age_group'] = pd.cut(
    clients_experiment_df['client_age'],
    labels = labels,
    bins = bins,
    include_lowest = True
)

clients_df['age_group'].value_counts()

age_group
35-54    26078
55-74    22221
19-34    19762
>74       1792
0-18       742
Name: count, dtype: int64

In [23]:
gender_age_counts = (
    clients_experiment_df
    .groupby(
        ['Variation','gender','age_group'],
        observed = False
    )['gender']
    .count()
    .sort_values(ascending = False)
)

gender_age_counts

Variation  gender  age_group
Test       U       19-34        3819
           F       35-54        3476
           M       35-54        3469
           F       55-74        3312
Control    U       19-34        3257
Test       M       55-74        3219
Control    M       35-54        3069
           F       55-74        3004
           M       55-74        2940
Test       U       35-54        2917
Control    F       35-54        2906
           U       35-54        2565
Test       U       55-74        2295
Control    U       55-74        2008
Test       M       19-34        1941
Control    M       19-34        1634
Test       F       19-34        1623
Control    F       19-34        1360
Test       M       >74           325
Control    M       >74           301
Test       F       >74           287
Control    F       >74           263
Test       U       >74           143
Control    U       >74           113
Test       U       0-18           94
Control    U       0-18           71
         

In [24]:
clients_experiment_df.groupby(
    ['Variation','age_group'],
    observed = False
)['age_group'].count().sort_values(ascending = False)

Variation  age_group
Test       35-54        9862
           55-74        8826
Control    35-54        8540
           55-74        7952
Test       19-34        7383
Control    19-34        6251
Test       >74           755
Control    >74           677
Test       0-18          135
Control    0-18          107
Name: age_group, dtype: int64

In [25]:
clients_experiment_df.groupby(['Variation'])['client_age'].mean()

Variation
Control    47.256896
Test       46.885242
Name: client_age, dtype: float64

In [26]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['client_tenure_yr'].mean()

Variation  age_group
Control    0-18          8.822430
           19-34         9.181571
           35-54        12.218384
           55-74        13.834759
           >74          17.267356
Test       0-18          8.955556
           19-34         9.157524
           35-54        12.151389
           55-74        13.754136
           >74          17.246358
Name: client_tenure_yr, dtype: float64

In [27]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['balance'].mean()

Variation  age_group
Control    0-18          42231.362710
           19-34         70511.584065
           35-54        142941.165094
           55-74        211964.995726
           >74          267306.189970
Test       0-18          42117.520889
           19-34         70067.256814
           35-54        142414.217485
           55-74        216296.403126
           >74          237969.940464
Name: balance, dtype: float64

In [28]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['calls_6_month'].mean()

Variation  age_group
Control    0-18         2.121495
           19-34        3.210846
           35-54        2.888759
           55-74        3.299170
           >74          3.571640
Test       0-18         2.674074
           19-34        3.076527
           35-54        2.862401
           55-74        3.251530
           >74          3.378808
Name: calls_6_month, dtype: float64

In [29]:
clients_experiment_df.groupby(['Variation'])['logons_6_month'].mean()

Variation
Control    6.166277
Test       6.101851
Name: logons_6_month, dtype: float64

# 4. Performance Metrics

The redesign is evaluated using three required KPIs:

1. **Time spent on each process step**
2. **Error rate**
3. **Completion rate**

## 4.1 Time spent on each process step

In [30]:
web_experiment_df['DateTime'] = pd.to_datetime(
    web_experiment_df['date_time']
)

# Sort events by visit and timestamp
web_sorted_df = web_experiment_df.sort_values(
    by = ['visit_id', 'DateTime'],
    ascending = [True, True]
)

# Remove consecutive duplicate process steps within the same visit
web_sorted_df = web_sorted_df[
    ~(
        (web_sorted_df['visit_id'] == web_sorted_df['visit_id'].shift())
        &
        (web_sorted_df['process_step'] == web_sorted_df['process_step'].shift())
    )
]

# Calculate elapsed time between consecutive events
web_sorted_df['Time_Spent'] = (
    web_sorted_df
    .groupby('visit_id')['DateTime']
    .diff()
    .dt.total_seconds()
    .fillna(0)
)

# Calculate total recorded time for each step within each visit
visit_step_times_df = (
    web_sorted_df
    .groupby(
        ['visit_id', 'process_step']
    )['Time_Spent']
    .sum()
    .reset_index()
)

# Calculate the average recorded time for each process step
avg_step_times = (
    visit_step_times_df
    .groupby('process_step')['Time_Spent']
    .mean()
    .sort_values(ascending = False)
)

avg_step_times

process_step
confirm    109.089572
step_3     103.402551
step_1      79.231724
step_2      54.858959
start       49.250997
Name: Time_Spent, dtype: float64

In [31]:
# Reset the index after sorting and filtering
web_sorted_df.reset_index(
    inplace = True
)

web_sorted_df.drop(
    'index',
    axis = 1,
    inplace = True
)

## 4.2 Error rate

A move from a later process step to an earlier process step is treated as an error.

In [32]:
# Ensure events are ordered chronologically within each visit
web_sorted_df.sort_values(
    by = ['visit_id', 'DateTime'],
    inplace = True
)

# Define the expected process sequence
sequence = [
    'start',
    'step_1',
    'step_2',
    'step_3',
    'confirm'
]

# Map each process step to its position in the sequence
step_indices = {
    step: idx
    for idx, step in enumerate(sequence)
}

# Initialise the error indicator
web_sorted_df['error'] = 0

# Group events by visit
grouped = web_sorted_df.groupby(
    ['visit_id']
)

# Flag backwards movements
for _, group in grouped:
    steps = group['process_step'].tolist()

    for i in range(1, len(steps)):
        if step_indices[steps[i]] < step_indices[steps[i - 1]]:
            idx_current = group.index[i]
            web_sorted_df.loc[
                idx_current,
                'error'
            ] = 1

In [33]:
web_sorted_df['error'].value_counts()

error
0    258421
1     26082
Name: count, dtype: int64

In [34]:
# Original overall error-rate calculation retained
error_rate = 48842 / (235661 + 48842)

error_rate

0.17167481537980267

In [35]:
# Add experiment-group information to the cleaned web data
web_analysis_df = pd.merge(
    web_sorted_df,
    experiment_df,
    on = "client_id",
    how = "left"
)

web_analysis_df.isna().sum()

client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
DateTime        0
Time_Spent      0
error           0
Variation       0
dtype: int64

In [36]:
# Calculate total errors for Test and Control
test_error_sum = web_analysis_df[
    web_analysis_df['Variation'] == 'Test'
]['error'].sum()

control_error_sum = web_analysis_df[
    web_analysis_df['Variation'] == 'Control'
]['error'].sum()

print(
    "Total errors for Test group:",
    test_error_sum
)

print(
    "Total errors for Control group:",
    control_error_sum
)

Total errors for Test group: 16352
Total errors for Control group: 9730


In [37]:
# Calculate error rates for Test and Control
test_error_avg = (
    web_analysis_df[
        web_analysis_df['Variation'] == 'Test'
    ]['error'].sum()
    /
    web_analysis_df[
        web_analysis_df['Variation'] == 'Test'
    ]['date_time'].count()
)

control_error_avg = (
    web_analysis_df[
        web_analysis_df['Variation'] == 'Control'
    ]['error'].sum()
    /
    web_analysis_df[
        web_analysis_df['Variation'] == 'Control'
    ]['date_time'].count()
)

print(
    "Percentage of errors for Test website:",
    test_error_avg
)

print(
    "Percentage of errors for Control website:",
    control_error_avg
)

Percentage of errors for Test website: 0.10372672777442989
Percentage of errors for Control website: 0.07669993220766526


## 4.3 Completion rate

A visit is considered complete when it reaches the `confirm` step at least once. Each visit is therefore counted only once in the completion-rate numerator.

In [38]:
# Count unique completed visits
completion_sum_test = (
    web_analysis_df[
        (web_analysis_df['Variation'] == "Test")
        &
        (web_analysis_df['process_step'] == "confirm")
    ]['visit_id']
    .nunique()
)

completion_sum_control = (
    web_analysis_df[
        (web_analysis_df['Variation'] == "Control")
        &
        (web_analysis_df['process_step'] == "confirm")
    ]['visit_id']
    .nunique()
)

print(
    f"Number of completed visits for the Test website is: {completion_sum_test}"
)

print(
    f"Number of completed visits for the Control website is: {completion_sum_control}"
)

Number of completed visits for the Test website is: 21724
Number of completed visits for the Control website is: 16040


In [39]:
# Count total visits
num_of_visits_test = web_analysis_df[
    web_analysis_df['Variation'] == "Test"
]['visit_id'].nunique()

num_of_visits_control = web_analysis_df[
    web_analysis_df['Variation'] == "Control"
]['visit_id'].nunique()

print(
    f"Number of total visits for the Test website is: {num_of_visits_test}"
)

print(
    f"Number of total visits for the Control website is: {num_of_visits_control}"
)

Number of total visits for the Test website is: 37121
Number of total visits for the Control website is: 32182


In [40]:
# Calculate visit-level completion rates
completion_rate_test = (
    completion_sum_test
    /
    num_of_visits_test
)

completion_rate_control = (
    completion_sum_control
    /
    num_of_visits_control
)

print(
    f"The completion rate of the Test website: {completion_rate_test}"
)

print(
    f"The completion rate of the Control website: {completion_rate_control}"
)

The completion rate of the Test website: 0.5852213033054067
The completion rate of the Control website: 0.4984152631906034


## 4.4 KPI comparison

In [41]:
web_analysis_df.groupby(['Variation','process_step'])['Time_Spent'].mean()

Variation  process_step
Control    confirm         128.427817
           start            35.688781
           step_1           65.863034
           step_2           38.378099
           step_3           91.249920
Test       confirm          93.682535
           start            43.037955
           step_1           60.128823
           step_2           49.013194
           step_3           87.905789
Name: Time_Spent, dtype: float64

# 5. Hypothesis Testing

In [42]:
# Create a binary indicator for the confirm step
web_analysis_df['confirm'] = (
    web_analysis_df['process_step']
    .apply(
        lambda x: 1
        if x == 'confirm'
        else 0
    )
)

## 5.1 Is the Test completion rate higher?

- **H0:** `test_completion_rate <= control_completion_rate`
- **H1:** `test_completion_rate > control_completion_rate`

In [43]:
# Calculate completion status for each Test visit
test_visits_df = (
    web_analysis_df[
        web_analysis_df["Variation"] == "Test"
    ]
    .groupby(
        "visit_id"
    )["confirm"]
    .sum()
)

# Calculate completion status for each Control visit
control_visits_df = (
    web_analysis_df[
        web_analysis_df["Variation"] == "Control"
    ]
    .groupby(
        "visit_id"
    )["confirm"]
    .sum()
)

# Convert the results to dataframes
test_visits_df = pd.DataFrame(
    test_visits_df
).reset_index()

control_visits_df = pd.DataFrame(
    control_visits_df
).reset_index()

# Convert completion counts into binary outcomes
control_visits_df['confirm'] = (
    control_visits_df['confirm']
    .apply(
        lambda x: 1
        if x >= 1
        else 0
    )
)

test_visits_df['confirm'] = (
    test_visits_df['confirm']
    .apply(
        lambda x: 1
        if x >= 1
        else 0
    )
)

test_visits_df['confirm'].value_counts()

confirm
1    21724
0    15397
Name: count, dtype: int64

In [44]:
# Count completed visits
successes = [
    test_visits_df['confirm'].sum(),
    control_visits_df['confirm'].sum()
]

# Count total visits
nobs = [
    len(test_visits_df),
    len(control_visits_df)
]

# One-sided proportion z-test
z_stat, p_value = proportions_ztest(
    count = successes,
    nobs = nobs,
    alternative = 'larger'
)

print(
    f"Z-statistic: {z_stat}"
)

print(
    f"P-value: {p_value}"
)

alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis: "
        "The Test version has a significantly higher completion rate."
    )
else:
    print(
        "Fail to reject the null hypothesis: "
        "No significant improvement with the Test version."
    )

Z-statistic: 22.886498717542324
P-value: 3.166330008322444e-116
Reject the null hypothesis: The Test version has a significantly higher completion rate.


## 5.2 Does the Test version exceed the 5% cost-effectiveness threshold?

Vanguard requires the Test version to achieve at least a **5% relative increase** in completion rate.

- **H0:** `p_test / p_control <= 1.05`
- **H1:** `p_test / p_control > 1.05`

In [45]:
# Number of completed visits
test_success = test_visits_df[
    'confirm'
].sum()

control_success = control_visits_df[
    'confirm'
].sum()

# Total number of visits
test_total = len(
    test_visits_df
)

control_total = len(
    control_visits_df
)

# Completion rates
p_test = (
    test_success
    /
    test_total
)

p_control = (
    control_success
    /
    control_total
)

# Relative completion-rate ratio
risk_ratio = (
    p_test
    /
    p_control
)

relative_uplift = (
    risk_ratio
    -
    1
)

# Standard error of the log risk ratio
se_log_rr = np.sqrt(
    (1 / test_success)
    - (1 / test_total)
    + (1 / control_success)
    - (1 / control_total)
)

# Test the 5% relative threshold
threshold = 1.05

z_stat = (
    np.log(risk_ratio)
    -
    np.log(threshold)
) / se_log_rr

p_value = st.norm.sf(
    z_stat
)

print(
    f"Test completion rate: {p_test}"
)

print(
    f"Control completion rate: {p_control}"
)

print(
    f"Risk ratio: {risk_ratio}"
)

print(
    f"Relative uplift: {relative_uplift * 100}%"
)

print(
    f"Z-statistic: {z_stat}"
)

print(
    f"P-value: {p_value}"
)

alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis: "
        "The Test version exceeds the 5% relative completion-rate threshold."
    )
else:
    print(
        "Fail to reject the null hypothesis: "
        "There is not enough evidence that the Test version exceeds "
        "the 5% relative completion-rate threshold."
    )

Test completion rate: 0.5852213033054067
Control completion rate: 0.4984152631906034
Risk ratio: 1.1741640887141271
Relative uplift: 17.416408871412713%
Z-statistic: 15.74892540724382
P-value: 3.4929157939493477e-56
Reject the null hypothesis: The Test version exceeds the 5% relative completion-rate threshold.


## 5.3 Time comparison for completed visits

The following one-sided Welch t-test compares total recorded visit time for visits containing a `confirm` step.

- **H0:** `control_time <= test_time`
- **H1:** `control_time > test_time`

In [46]:
completed_visits_df = web_analysis_df.groupby(
    'visit_id'
).filter(
    lambda x:
    (x['process_step'] == 'confirm').any()
)

time_control = (
    completed_visits_df[
        completed_visits_df['Variation'] == "Control"
    ]
    .groupby(
        "visit_id"
    )['Time_Spent']
    .sum()
)

time_test = (
    completed_visits_df[
        completed_visits_df['Variation'] == "Test"
    ]
    .groupby(
        "visit_id"
    )['Time_Spent']
    .sum()
)

alpha = 0.05

st.ttest_ind(
    time_control,
    time_test,
    equal_var = False,
    alternative = "greater"
)

TtestResult(statistic=np.float64(13.458675304232976), pvalue=np.float64(1.733997172609756e-41), df=np.float64(34956.563968805625))

# 6. Conclusion

The final evaluation should consider all three KPIs together with the hypothesis-test results:

- Compare the Test and Control **visit-level completion rates**.
- Determine whether the Test version has a statistically higher completion rate.
- Determine whether the relative improvement exceeds Vanguard's required **5% threshold**.
- Compare the Test and Control error rates.
- Compare the recorded time results between the two versions.

The completion-rate hypothesis tests answer two different questions: whether completion improved at all, and whether the improvement was large enough to exceed the business threshold.

# 7. Optional Export Commands

The project does not save processed datasets during normal execution. The following export commands are retained only as reference.

In [47]:
# web_analysis_df.to_csv(
#     "sorted_df_web_cleaned_exp.csv",
#     index = False
# )

# completed_visits_df.to_csv(
#     "filtered_df.csv",
#     index = False
# )

# clients_experiment_df.to_csv(
#     "df_client_exp.csv",
#     index = False
# )